# 06 · Context-ablation timing: instruction flag on/off

Measures how long **one** context-ablation pass takes for a single QA, and
whether folding the generation system prompt into the ablation set (the
`instruction` flag) changes the wall-clock time.

**Design choices:**
- **In-process via `attribute_one`, not the HTTP API.** `POST /attribute` is
  hardwired to a `CachingProvider` and never passes `instruction`, so it can
  satisfy neither requirement here. We call `attribute_one()` (a drop-in for the
  engine `attribute_by_sentence`, in `tools/attribute_one.py`) directly instead —
  it's also the sandbox where ablation concurrency will be prototyped later.
- **No cache.** We use a bare `OpenAICompatProvider` (never wrapped in
  `CachingProvider`), so every ablation call hits NIM fresh. Timing is real,
  not a replay of `./.cache/logprobs.sqlite`.

Run order: setup → pick QA → knobs → **Cell A (no instruction)** → **Cell B
(with instruction)** → compare.

## Setup — non-caching provider

In [ ]:
import sys, time
from pathlib import Path

# Put the repo root (parent of this notebooks/ dir) on sys.path so `tools` and
# `dare` import regardless of cwd/kernel. `dare` is pip-installed (editable) so it
# would resolve anyway, but `tools` is NOT a package on the path — it's just a repo
# directory — so this line is what makes `from tools...` work.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dare.config import Settings
from dare.providers import OpenAICompatProvider
from dare.schema import load_corpus
from dare.prompts import SYNTHESIS_SYSTEM

# attribute_one is a drop-in for dare.attribution.attribute_by_sentence (same args,
# same {response, whole, units} return). It delegates to the engine today; it's the
# sandbox where ablation concurrency gets prototyped later, so the cells below don't
# change when that lands. See tools/attribute_one.py.
from tools.attribute_one import attribute_one

settings = Settings.from_env()  # loads repo-root .env regardless of cwd

def fresh_provider():
    """A bare provider with NO caching wrapper — every score_response() call goes
    to NIM. A fresh instance per run so nothing (even in-memory) carries over."""
    return OpenAICompatProvider(settings)

print(f"repo root  : {REPO_ROOT}")
print(f"model      : {settings.model}")
print(f"llm_url    : {settings.llm_url}")
print(f"call delay : {getattr(settings, 'api_call_delay', '?')}s between ablation calls (applies equally to both runs)")
print("caching    : OFF (bare OpenAICompatProvider)")

## Pick which QA to run

Set `QA_ID` to a record id, or leave it `None` and use `QA_INDEX`.

In [ ]:
CORPUS = "../data/raw_responses.jsonl"
QA_ID = None      # e.g. "qa-luongvh-0001"; if None, falls back to QA_INDEX
QA_INDEX = 0      # used only when QA_ID is None

records = load_corpus(CORPUS)

if QA_ID is not None:
    rec = next(r for r in records if r.id == QA_ID)
else:
    rec = records[QA_INDEX]

assert rec.attributable, f"record {rec.id!r} has no answer/sources to attribute"

print(f"id      : {rec.id}")
print(f"chunks  : {len(rec.chunks)}")
print(f"query   : {rec.query}")
print(f"answer  : {rec.answer[:300]}{'...' if len(rec.answer) > 300 else ''}")
print(f"\n(pick another: set QA_ID, or QA_INDEX in 0..{len(records) - 1})")

## Knobs

`NUM_ABLATIONS` drives the API-call count (dominant cost). `INSTRUCTION` is the
text folded in for Cell B — defaults to the real generation system prompt.

In [ ]:
NUM_ABLATIONS = 32          # ~this many sequential NIM calls per run
INSTRUCTION = SYNTHESIS_SYSTEM   # edit to test any instruction text

print(f"NUM_ABLATIONS = {NUM_ABLATIONS}")
print(f"INSTRUCTION   = {INSTRUCTION[:120]}...")

## Cell A — ablation WITHOUT the instruction flag

In [ ]:
t0 = time.perf_counter()
result_no_instr = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # no cache
    settings=settings,
    # instruction omitted -> context-only ablation
)
elapsed_no_instr = time.perf_counter() - t0

print(f"WITHOUT instruction: {elapsed_no_instr:.2f}s  ({len(rec.chunks)} chunks, {NUM_ABLATIONS} ablations, {len(result_no_instr['units'])} units)")

## Cell B — ablation WITH the instruction flag

Same QA, same ablation count; the instruction is prepended into the context as
its own lane, so the ablation prompt is slightly longer per call.

In [ ]:
t0 = time.perf_counter()
result_with_instr = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # no cache
    settings=settings,
    instruction=INSTRUCTION,     # folded in -> origin='instruction' lane
)
elapsed_with_instr = time.perf_counter() - t0

print(f"WITH instruction   : {elapsed_with_instr:.2f}s  ({len(rec.chunks)} chunks, {NUM_ABLATIONS} ablations, {len(result_with_instr['units'])} units)")

## Compare

In [ ]:
delta = elapsed_with_instr - elapsed_no_instr
pct = (delta / elapsed_no_instr * 100) if elapsed_no_instr else float('nan')

print(f"QA            : {rec.id}")
print(f"ablations     : {NUM_ABLATIONS}   chunks: {len(rec.chunks)}   cache: OFF")
print(f"without instr : {elapsed_no_instr:7.2f}s")
print(f"with instr    : {elapsed_with_instr:7.2f}s")
print(f"delta         : {delta:+7.2f}s  ({pct:+.1f}%)")

## Concurrency verification — pin the logprobs, then serial vs concurrent must match

The earlier un-cached run proved concurrency is **fast** (real ~3.7× speedup) but
its scores differed from serial. That difference is **model variability**, not our
code: two separate live runs get different logprobs from NIM (temperature-0 serving
isn't bit-reproducible, and concurrent requests can be batched server-side into
slightly different numerics). An un-cached comparison therefore tests *NIM
determinism*, not *our concurrency*.

To verify the **code** (does concurrency put each logprob in the right slot?), we
must hold the logprobs constant. Two steps:

1. **NIM-determinism probe** (next cell) — score the same input twice and diff the
   logprobs. If it's > 0, model variability is confirmed and caching is required.
2. **Cached serial vs concurrent** — one *shared* `CachingProvider` for both runs:
   the serial pass populates the cache, the concurrent pass **replays the identical
   logprobs**. Now the only thing that can differ is our concurrency code, so the
   scores **must** match exactly. If they don't, it's a real pairing bug.

(Timing in the cached cells is not the speedup — the replay run hits cache, not the
network. The real speedup is the un-cached run above.)

In [ ]:
# --- NIM-determinism probe: score the SAME input twice, diff the logprobs ---
from dare.schema import chunks_to_context

probe = fresh_provider()   # uncached, live
probe_user = chunks_to_context(rec.chunks) + "\n\n" + rec.query
probe_resp = rec.answer

a = probe.score_response(probe_user, probe_resp)
b = probe.score_response(probe_user, probe_resp)

assert len(a) == len(b), f"token count differs across calls: {len(a)} vs {len(b)}"
max_lp_diff = max((abs(x[1] - y[1]) for x, y in zip(a, b)), default=0.0)
print(f"scored {len(a)} response tokens, twice, on IDENTICAL input")
print(f"max |logprob diff| between the two calls : {max_lp_diff:.3e}")
print()
if max_lp_diff == 0:
    print("NIM is DETERMINISTIC per call -> live runs should already match; caching still")
    print("recommended below to hold logprobs fixed while isolating the concurrency code.")
else:
    print("NIM is NON-DETERMINISTIC per call -> THIS is the model variability behind the")
    print("earlier FAIL. Pin logprobs with the shared cache in the cells below to verify")
    print("the concurrency code itself.")

In [ ]:
from dare.providers import CachingProvider

MAX_WORKERS = 8   # concurrent ablation calls for the second run (start modest; watch for 429s)

# ONE shared caching provider for BOTH runs. This serial pass populates the cache;
# the concurrent pass below replays the SAME logprobs -> model variability is held
# fixed, so any score difference is purely our concurrency code.
shared = CachingProvider(OpenAICompatProvider(settings))

# --- SERIAL run: max_workers=1 (today's path). Populates the cache. ---
t0 = time.perf_counter()
res_serial = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=shared,             # cache warms here
    settings=settings,
    max_workers=1,               # serial baseline
)
elapsed_serial = time.perf_counter() - t0

serial_misses = shared.misses    # snapshot: concurrent run should add 0 more misses
scores_serial = [round(a.score, 4) for a in res_serial["whole"]]
print(f"SERIAL (warms cache) : {elapsed_serial:6.2f}s   cache: {shared.hits} hits, {shared.misses} misses")
print(f"  whole scores = {scores_serial}")

In [ ]:
# --- CONCURRENT run: same `shared` provider -> cache HITS, identical logprobs replayed ---
t0 = time.perf_counter()
res_conc = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=shared,             # SAME cache -> replays serial's logprobs (no NIM variability)
    settings=settings,
    max_workers=MAX_WORKERS,     # concurrent ablation calls
)
elapsed_conc = time.perf_counter() - t0

# Fresh misses added by THIS run must be 0 — same masks -> same user_contents -> all hits.
conc_new_misses = shared.misses - serial_misses
scores_conc = [round(a.score, 4) for a in res_conc["whole"]]
print(f"CONCURRENT (replays cache) : {elapsed_conc:6.2f}s   cache: {shared.hits} hits, +{conc_new_misses} new misses  (max_workers={MAX_WORKERS})")
print(f"  whole scores = {scores_conc}")

In [ ]:
import numpy as np

# whole-response scores, serial vs concurrent — logprobs pinned by the shared cache
s = np.array([a.score for a in res_serial["whole"]])
c = np.array([a.score for a in res_conc["whole"]])
whole_ok = s.shape == c.shape and np.allclose(s, c, atol=1e-6)

# every unit's scores must match too
unit_ok = all(
    np.allclose([x.score for x in us.attributions],
                [x.score for x in uc.attributions], atol=1e-6)
    for us, uc in zip(res_serial["units"], res_conc["units"])
)

# Sanity: the concurrent run added NO fresh misses -> it truly replayed serial's
# logprobs. If it did miss, the logprobs weren't pinned and the test is meaningless.
pinned = conc_new_misses == 0

print(f"whole max abs diff : {np.max(np.abs(s - c)):.2e}")
print(f"whole identical    : {whole_ok}")
print(f"units identical    : {unit_ok}")
print(f"logprobs pinned    : {pinned}  (concurrent run added {conc_new_misses} new cache misses; expect 0)")
print()
if pinned and whole_ok and unit_ok:
    print("PASS - with logprobs held fixed, concurrency == serial exactly.")
    print("       Pairing is correct; the concurrency code is safe to ship.")
elif not pinned:
    print("INCONCLUSIVE - concurrent run wasn't pure cache replay, so logprobs weren't")
    print("       pinned. Restart + run the two cells above in order (serial then concurrent).")
else:
    print("FAIL - identical logprobs yet different scores -> a real concurrency bug in the")
    print("       code (mask<->logprob pairing). Do NOT ship; investigate APIModel.__call__.")

# Note: wall-clock above is NOT the speedup (concurrent run is a cache replay).
# The real ~3.7x speedup is the earlier un-cached serial-vs-concurrent run.

## Reverse test — REAL cold concurrent timing, then serial replay

The concurrency-verification cells above run **serial first (warms cache) → concurrent (replays cache)**, so the concurrent timing there is a cache *replay*, not a real speedup. This does the **reverse** on a guaranteed-fresh cache:

1. **Cold concurrent** (`max_workers=MAX_WORKERS`, empty temp cache) — every ablation is a live NIM call, run concurrently. **This wall-clock is the real number**: it's exactly what the live widget does (instruction folded, cold, concurrent via `api.py`'s `widget_max_workers`).
2. **Serial replay** on that now-warm cache (`max_workers=1`) — all cache hits, 0 new misses; used only to confirm the concurrent scores are correct (logprobs pinned by the cold run).

The true cold speedup is **Cell B (cold serial, instruction) ÷ the cold concurrent time below** — both hit NIM live for the same QA, so it's apples-to-apples.

In [ ]:
import tempfile
from pathlib import Path
from dare.providers import CachingProvider, OpenAICompatProvider

MAX_WORKERS = 8   # concurrent ablation calls -- mirrors api.py's widget_max_workers default

# Guaranteed-COLD cache: a fresh sqlite path that doesn't exist yet, so every ablation
# misses -> a real live NIM call (NOT a replay of ./.cache/logprobs.sqlite).
COLD_DB = Path(tempfile.mkdtemp(prefix="dare_cold_")) / "cold.sqlite"
cold = CachingProvider(OpenAICompatProvider(settings), path=COLD_DB)

# Instruction folded in, matching the live endpoint (api.py always folds it).
t0 = time.perf_counter()
res_cold_conc = attribute_one(
    rec.query, rec.answer, rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=cold,
    settings=settings,
    instruction=INSTRUCTION,
    max_workers=MAX_WORKERS,
)
elapsed_cold_conc = time.perf_counter() - t0
cold_misses_after_conc = cold.misses

print(f"COLD CONCURRENT (live NIM, max_workers={MAX_WORKERS}) : {elapsed_cold_conc:6.2f}s")
print(f"  cache: {cold.hits} hits, {cold.misses} misses  (expect ~{NUM_ABLATIONS} misses = all live)")
print(f"  fresh cache: {COLD_DB}")
print("  -> this is the real live-widget latency profile (instruction folded, cold, concurrent)")


In [ ]:
import numpy as np

# --- SERIAL replay on the now-warm cold cache: max_workers=1, all cache HITS ---
# Deterministic masks -> same user_contents -> this replays the cold run's logprobs,
# so any score difference would be a concurrency pairing bug (not model variability).
t0 = time.perf_counter()
res_cold_serial = attribute_one(
    rec.query, rec.answer, rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=cold,               # SAME provider -> replays the cold-concurrent logprobs
    settings=settings,
    instruction=INSTRUCTION,
    max_workers=1,               # serial
)
elapsed_cold_serial = time.perf_counter() - t0
serial_new_misses = cold.misses - cold_misses_after_conc   # expect 0 -> pure cache replay

a = np.array([x.score for x in res_cold_conc["whole"]])
b = np.array([x.score for x in res_cold_serial["whole"]])
scores_match = a.shape == b.shape and np.allclose(a, b, atol=1e-6)

print(f"COLD CONCURRENT (live, above)     : {elapsed_cold_conc:6.2f}s")
print(f"SERIAL replay (cache hits)        : {elapsed_cold_serial:6.2f}s   (+{serial_new_misses} new misses, expect 0)")
print(f"scores identical (conc == serial) : {scores_match}   max|diff| = {np.max(np.abs(a - b)):.2e}")
print()
try:
    speedup = elapsed_with_instr / elapsed_cold_conc
    print("TRUE COLD SPEEDUP (instruction folded, same QA, both live):")
    print(f"  cold SERIAL   (Cell B)          : {elapsed_with_instr:6.2f}s")
    print(f"  cold CONCURRENT (max_workers={MAX_WORKERS})  : {elapsed_cold_conc:6.2f}s")
    print(f"  speedup                         : {speedup:.2f}x")
except NameError:
    print("(run Cell B above for the cold-serial baseline, then re-run this cell for the speedup ratio)")
